# Ray forward and backward pair mathing
Fresnel zone calculation requires forward and backward ray seperation $\zeta_1$ and $\zeta_2$. <br>
For the case where there are multiple eigen rays with the same ray_id, the forward and backward ray paths are hand matched.

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
from dotenv import find_dotenv, load_dotenv

from nescafe23_fluctuations import tools

_ = load_dotenv(find_dotenv())

%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

In [ ]:
dss = tools.open_receptions()

In [ ]:
dss[0].nominal_depth

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_run_forward_back.pkl"

with open(fn, "rb") as f:
    ray_run = pickle.load(f)

eigs = ray_run["eigs"]
eigs_back = ray_run["eigs_back"]
envi = ray_run["envi"]
envi_back = ray_run["envi_back"]


In [ ]:
def plot_forward_backward_rays(ray_id, front_idx, back_idx):
    """
    plot forward and backward rays to get proper pairs

    Parameters
    ----------
    ray_id : str
        ray id
    front_idx : int
        relative index of forward ray
    back_idx : int
        relative index of backward ray
    """
    ray_id_pr, (sign, number, suffix) = tools.parse_ray_id(ray_id)

    # Determine backward ray_id (flip sign if even number of turning points)
    if number % 2 != 0:
        ray_id_back = ray_id
    else:
        sign_back = "+" if sign == "-" else "-"
        ray_id_back = f"{sign_back}{number}{suffix}"

    # Convert backward ray_id to pr format
    ray_id_pr_back, _ = tools.parse_ray_id(ray_id_back)

    print(ray_id_pr, ray_id_pr_back)
    print(f"Number of forward rays: {len(np.where(eigs.ray_id[0] == ray_id_pr)[0])}")
    print(
        f"Number of backward rays: {len(np.where(eigs_back.ray_id[0] == ray_id_pr_back)[0])}"
    )

    try:
        ray_idx_forward = np.where(eigs.ray_id[0] == ray_id_pr)[0][front_idx]
    except IndexError:
        print("❌ Index Error")
        ray_idx_forward = np.where(eigs.ray_id[0] == ray_id_pr)[0]

    try:
        ray_idx_backward = np.where(eigs_back.ray_id[0] == ray_id_pr_back)[0][back_idx]
    except IndexError:
        print("❌ Index Error")
        ray_idx_backward = np.where(eigs_back.ray_id[0] == ray_id_pr_back)[0]

    receiver_range = eigs.rs[0][ray_idx_forward, -1]
    fig = plt.figure()
    
    _ = plt.plot(
        eigs.rs[0][ray_idx_forward, :].T, eigs.zs[0][ray_idx_forward, :].T, c="k"
    )
    _ = plt.plot(
        -eigs_back.rs[0][ray_idx_backward, :].T + receiver_range,
        eigs_back.zs[0][ray_idx_backward, :].T,
        c="r",
    )

    plt.title(f"Ray ID: {ray_id}")

In [ ]:
def plot_sameid(ray_id):
    ray_id_pr, (sign, number, suffix) = tools.parse_ray_id(ray_id)
    ray_idx_forward = np.where(eigs.ray_id[0] == ray_id_pr)[0]

    receiver_range = eigs.rs[0][ray_idx_forward, -1]

    _ = plt.plot(
        eigs.rs[0][ray_idx_forward, :].T, eigs.zs[0][ray_idx_forward, :].T,
    )

In [ ]:
plot_sameid('-7')
plot_sameid('+6')
plot_sameid('-6')
plot_sameid('+7')

### hand pick the indices

In [ ]:
eig_indexs = {
    "+7": [(0, 0), (1,1)],
    "-7": [(0, 2), (1,1), (2,0)],
    "-6": [(0, 0)],
    "+6": [(0, 0)],
    "-5b": [(0, 0), (1,1)],
    "-6b": [(0, 0)],
    "+6b": [(0, 0)],
    "+7b": [(0, 0)],
    "-7b": [(0, 0)],
    "-8b": [(0, 0)],
    "+8b": [(0, 0)],
    "+9b": [(1, 0)],
    "-9b": [(0, 0)],
    "-10b": [(0, 0)],
    "+10b": [(0, 0)],
    "+11b": [(0, 0)],
    "-11b": [(0, 0)],
    "-12b": [(0, 0)],
    "+12b": [(0, 0)],
    "+13b": [(0, 0)],
    "-13b": [(0, 0)],
    "-14b": [(0, 0)],
    "+14b": [(0, 0)],
    "+15b": [(0, 0)],
    "-15b": [(0, 0)],
}

key_idx = 6
keys = list(eig_indexs.keys())

ray_id = keys[key_idx]

pair_idx = 0
front_idx = eig_indexs[keys[key_idx]][pair_idx][0] # [arrival pair, forward/backward]
back_idx = eig_indexs[keys[key_idx]][pair_idx][1]

plot_forward_backward_rays(ray_id, front_idx, back_idx)

### Plot all eigen ray pairs

In [ ]:
# Hand choosing the eigenray indices so that forward / backward match, and choosing
# the paths that we actually want

for key_idx in range(24):
    keys = list(eig_indexs.keys())

    front_idx = eig_indexs[keys[key_idx]][0][0]
    back_idx = eig_indexs[keys[key_idx]][0][1]

    ray_id = keys[key_idx]
    plot_forward_backward_rays(ray_id, front_idx, back_idx)